In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import os
from functools import reduce
# PATH
folder = "/content/drive/MyDrive/Dataset"
# LOAD DATASETS
logon = pd.read_csv(os.path.join(folder,"logon.csv"), low_memory=False)
device = pd.read_csv(os.path.join(folder,"device.csv"), low_memory=False)
file = pd.read_csv(os.path.join(folder,"file.csv"), low_memory=False)
email = pd.read_csv(os.path.join(folder,"email.csv"), low_memory=False)
https = pd.read_csv(os.path.join(folder,"https.csv"), low_memory=False)
psy = pd.read_csv(os.path.join(folder,"psychometric.csv"), low_memory=False)
# CONVERT DATE
datasets = [logon, device, file, email, https]
for df in datasets:
    df["date"] = pd.to_datetime(df["date"])
    df["hour_window"] = df["date"].dt.floor("H")
# LOGON FEATURES
logon_features = (
    logon
    .groupby(["user","hour_window"])
    .agg(
        login_events=("activity","count")
    )
    .reset_index()
)
# DEVICE FEATURES
device_features = (
    device
    .groupby(["user","hour_window"])
    .agg(
        device_events=("activity","count")
    )
    .reset_index()
)
# FILE FEATURES
file_features = (
    file
    .groupby(["user","hour_window"])
    .agg(
        file_events=("filename","count")
    )
    .reset_index()
)
# EMAIL FEATURES
email_features = (
    email
    .groupby(["user","hour_window"])
    .agg(
        email_events=("to","count"),
        total_email_size=("size","sum"),
        total_attachments=("attachments","sum")
    )
    .reset_index()
)
# HTTPS FEATURES
https_features = (
    https
    .groupby(["user","hour_window"])
    .agg(
        https_events=("url","count")
    )
    .reset_index()
)
# MERGE ALL LOGS
dfs = [
    logon_features,
    device_features,
    file_features,
    email_features,
    https_features
]

final_df = reduce(
    lambda left,right:
    pd.merge(
        left,
        right,
        on=["user","hour_window"],
        how="outer"
    ),
    dfs
)
# MERGE PSYCHOMETRIC
final_df = final_df.merge(
    psy,
    left_on="user",
    right_on="user_id",
    how="left"
)
final_df.drop(
    columns=["employee_name","user_id"],
    inplace=True,
    errors="ignore"
)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_4862/510256944.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour_window"] = df["date"].dt.floor("H")
/tmp/ipykernel_4862/510256944.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour_window"] = df["date"].dt.floor("H")
/tmp/ipykernel_4862/510256944.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour_window"] = df["date"].dt.floor("H")
/tmp/ipykernel_4862/510256944.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour_window"] = df["date"].dt.floor("H")
/tmp/ipykernel_4862/510256944.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["hour_window"] = df["date"].dt.floor("H")


In [ ]:
# FILL MISSING VALUES
numeric_cols = final_df.select_dtypes(include=["number"]).columns
final_df[numeric_cols] = final_df[numeric_cols].fillna(0)
# RESULT
print("="*80)
print("FINAL DATASET")
print("="*80)
print(final_df.shape)
print("\nColumns\n")
print(final_df.columns.tolist())
print("\nMissing Values\n")
print(final_df.isnull().sum())
print("\nPreview\n")
print(final_df.head())

FINAL DATASET
(1621209, 14)

Columns

['user', 'hour_window', 'login_events', 'device_events', 'file_events', 'email_events', 'total_email_size', 'total_attachments', 'https_events', 'O', 'C', 'E', 'A', 'N']

Missing Values

user                 0
hour_window          0
login_events         0
device_events        0
file_events          0
email_events         0
total_email_size     0
total_attachments    0
https_events         0
O                    0
C                    0
E                    0
A                    0
N                    0
dtype: int64

Preview

      user         hour_window  login_events  device_events  file_events  \
0  AAE0190 2010-01-04 08:00:00           1.0            0.0          0.0   
1  AAE0190 2010-01-04 09:00:00           0.0            0.0          0.0   
2  AAE0190 2010-01-04 10:00:00           0.0            0.0          0.0   
3  AAE0190 2010-01-04 11:00:00           0.0            0.0          0.0   
4  AAE0190 2010-01-04 12:00:00           0.0      

In [ ]:
print("="*80)
print("FINAL DATASET INFORMATION")
print("="*80)
print("Shape :", final_df.shape)
print("\nColumns:")
print(final_df.columns.tolist())
print("\nData Types:")
print(final_df.dtypes)
print("\nMissing Values:")
print(final_df.isnull().sum())
print("\nDuplicate Rows:", final_df.duplicated().sum())

FINAL DATASET INFORMATION
Shape : (1621209, 14)

Columns:
['user', 'hour_window', 'login_events', 'device_events', 'file_events', 'email_events', 'total_email_size', 'total_attachments', 'https_events', 'O', 'C', 'E', 'A', 'N']

Data Types:
user                         object
hour_window          datetime64[ns]
login_events                float64
device_events               float64
file_events                 float64
email_events                float64
total_email_size            float64
total_attachments           float64
https_events                float64
O                             int64
C                             int64
E                             int64
A                             int64
N                             int64
dtype: object

Missing Values:
user                 0
hour_window          0
login_events         0
device_events        0
file_events          0
email_events         0
total_email_size     0
total_attachments    0
https_events         0
O                

In [ ]:
print("="*80)
print("FIRST 10 ROWS")
print("="*80)
display(final_df.head(10))
print("="*80)
print("LAST 10 ROWS")
print("="*80)
display(final_df.tail(10))

FIRST 10 ROWS


,user,hour_window,login_events,device_events,file_events,email_events,total_email_size,total_attachments,https_events,O,C,E,A,N
0,AAE0190,2010-01-04 08:00:00,1.0,0.0,0.0,13.0,404826.0,4.0,9.0,36,30,14,50,29
1,AAE0190,2010-01-04 09:00:00,0.0,0.0,0.0,1.0,36502.0,0.0,20.0,36,30,14,50,29
2,AAE0190,2010-01-04 10:00:00,0.0,0.0,0.0,0.0,0.0,0.0,15.0,36,30,14,50,29
3,AAE0190,2010-01-04 11:00:00,0.0,0.0,0.0,0.0,0.0,0.0,18.0,36,30,14,50,29
4,AAE0190,2010-01-04 12:00:00,0.0,0.0,0.0,0.0,0.0,0.0,9.0,36,30,14,50,29
5,AAE0190,2010-01-04 18:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,36,30,14,50,29
6,AAE0190,2010-01-05 08:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,36,30,14,50,29
7,AAE0190,2010-01-05 09:00:00,0.0,0.0,0.0,6.0,150298.0,0.0,0.0,36,30,14,50,29
8,AAE0190,2010-01-05 10:00:00,0.0,0.0,0.0,7.0,205254.0,2.0,0.0,36,30,14,50,29
9,AAE0190,2010-01-05 18:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,36,30,14,50,29


LAST 10 ROWS


,user,hour_window,login_events,device_events,file_events,email_events,total_email_size,total_attachments,https_events,O,C,E,A,N
1621199,ZSL0305,2011-05-11 11:00:00,0.0,0.0,0.0,1.0,46870.0,0.0,0.0,21,26,18,41,32
1621200,ZSL0305,2011-05-11 17:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,21,26,18,41,32
1621201,ZSL0305,2011-05-12 08:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,21,26,18,41,32
1621202,ZSL0305,2011-05-12 12:00:00,0.0,0.0,0.0,1.0,28961.0,0.0,0.0,21,26,18,41,32
1621203,ZSL0305,2011-05-12 17:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,21,26,18,41,32
1621204,ZSL0305,2011-05-13 09:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,21,26,18,41,32
1621205,ZSL0305,2011-05-13 17:00:00,1.0,0.0,0.0,1.0,26769.0,0.0,0.0,21,26,18,41,32
1621206,ZSL0305,2011-05-16 09:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,21,26,18,41,32
1621207,ZSL0305,2011-05-16 11:00:00,0.0,0.0,0.0,1.0,25610.0,0.0,0.0,21,26,18,41,32
1621208,ZSL0305,2011-05-16 17:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,21,26,18,41,32
